# RiskFlow PayGuard — IEEE-CIS Fraud-Risk EDA

## Phase 1 objective

This notebook examines the IEEE-CIS Fraud Detection dataset from the perspective
of a production fraud-risk product.

The analysis focuses on:

- source-table contracts
- target imbalance
- identity-data coverage
- missingness and cardinality
- transaction amount behavior
- chronological fraud patterns
- fraud-rate segmentation
- baseline feature selection

Raw Kaggle files remain local and are not committed to Git.


## 1. Setup

The transaction table is the primary table. Identity data is optional and is
linked through `TransactionID`.

The complete transaction and identity tables are loaded separately in this
section. A full wide join is intentionally avoided until it is analytically
necessary.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

from src.data_processing import (
    AMOUNT_COLUMN,
    JOIN_KEY,
    TARGET_COLUMN,
    TIME_COLUMN,
    load_train_tables,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Raw data:     {DATA_DIR.relative_to(PROJECT_ROOT)}")
print(f"Processed:    {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")

Project root: riskflow-payguard
Raw data:     data/raw
Processed:    data/processed


## 2. Load the training source tables

Set `NROWS` to an integer such as `100_000` for quick development.

Use `None` for the complete Phase 1 analysis.


In [2]:
NROWS: int | None = None

transaction, identity = load_train_tables(
    DATA_DIR,
    nrows=NROWS,
)

print(f"Transaction shape: {transaction.shape}")
print(f"Identity shape:    {identity.shape}")

Transaction shape: (590540, 394)
Identity shape:    (144233, 41)


## 3. Dataset dimensions and join coverage

Identity attributes exist for only a subset of transactions. Their absence may
be operationally meaningful, so identity availability should be measured before
deciding how missing identity values will be handled.


In [3]:
transaction_memory_mb = (
    transaction.memory_usage(deep=True).sum() / 1024**2
)
identity_memory_mb = (
    identity.memory_usage(deep=True).sum() / 1024**2
)

identity_key_set = set(identity[JOIN_KEY])
has_identity = transaction[JOIN_KEY].isin(identity_key_set)

identity_coverage = has_identity.mean()
orphan_identity_count = int(
    (~identity[JOIN_KEY].isin(transaction[JOIN_KEY])).sum()
)

dataset_dimensions = pd.DataFrame(
    [
        {
            "table": "train_transaction",
            "rows": len(transaction),
            "columns": transaction.shape[1],
            "unique_transaction_ids": transaction[JOIN_KEY].nunique(),
            "memory_mb": transaction_memory_mb,
        },
        {
            "table": "train_identity",
            "rows": len(identity),
            "columns": identity.shape[1],
            "unique_transaction_ids": identity[JOIN_KEY].nunique(),
            "memory_mb": identity_memory_mb,
        },
    ]
)

display(dataset_dimensions)

print(f"Identity coverage:       {identity_coverage:.2%}")
print(f"Transactions with ID:    {has_identity.sum():,}")
print(f"Transactions without ID: {(~has_identity).sum():,}")
print(f"Orphan identity rows:    {orphan_identity_count:,}")

,table,rows,columns,unique_transaction_ids,memory_mb
0,train_transaction,590540,394,590540,"1,791.6674"
1,train_identity,144233,41,144233,56.5075


Identity coverage:       24.42%
Transactions with ID:    144,233
Transactions without ID: 446,307
Orphan identity rows:    0


In [4]:
assert transaction[JOIN_KEY].is_unique
assert identity[JOIN_KEY].is_unique
assert orphan_identity_count == 0
assert TARGET_COLUMN in transaction.columns
assert transaction[TARGET_COLUMN].notna().all()
assert set(transaction[TARGET_COLUMN].unique()).issubset({0, 1})

print("Source-table contract checks passed.")

Source-table contract checks passed.


## 4. Target imbalance

Fraud detection is an imbalanced classification problem. Accuracy would provide
a misleading view of model quality because legitimate transactions dominate the
dataset.

The initial model evaluation will therefore emphasize ranking, probability
quality, and fraud-capture metrics rather than accuracy.


In [5]:
target_counts = (
    transaction[TARGET_COLUMN]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

legitimate_count = int(target_counts.loc[0])
fraud_count = int(target_counts.loc[1])
total_count = int(target_counts.sum())

fraud_rate = fraud_count / total_count
legitimate_to_fraud_ratio = (
    legitimate_count / fraud_count if fraud_count else float("inf")
)

target_summary = pd.DataFrame(
    [
        {
            "class": "legitimate",
            "target_value": 0,
            "transaction_count": legitimate_count,
            "share": legitimate_count / total_count,
        },
        {
            "class": "fraud",
            "target_value": 1,
            "transaction_count": fraud_count,
            "share": fraud_rate,
        },
    ]
)

display(target_summary)

print(f"Total transactions:          {total_count:,}")
print(f"Fraudulent transactions:     {fraud_count:,}")
print(f"Overall fraud rate:          {fraud_rate:.4%}")
print(
    "Legitimate-to-fraud ratio: "
    f"{legitimate_to_fraud_ratio:,.1f}:1"
)

,class,target_value,transaction_count,share
0,legitimate,0,569877,0.9650
1,fraud,1,20663,0.0350


Total transactions:          590,540
Fraudulent transactions:     20,663
Overall fraud rate:          3.4990%
Legitimate-to-fraud ratio: 27.6:1


### Initial product implication

The observed imbalance will inform:

- `scale_pos_weight` or equivalent class weighting
- PR AUC and ROC AUC reporting
- precision and recall at decision thresholds
- manual-review capacity analysis
- fraud-capture and false-positive trade-offs

Resampling methods such as SMOTE are deferred until the chronological baseline
has been evaluated.


## 5. Missingness, data types, and cardinality

The IEEE-CIS dataset contains hundreds of sparse, anonymous, categorical, and
identifier-like fields.

This section creates a reusable profile for each source column without filling
missing values or joining the full transaction and identity tables.

The profile records:

- data type
- missing count and percentage
- non-null count
- distinct-value count
- distinct-value ratio
- cardinality band
- initial modeling-role hint


In [6]:
def classify_missingness(missing_pct: float) -> str:
    if missing_pct == 0:
        return "complete"
    if missing_pct <= 10:
        return "low_0_10"
    if missing_pct <= 50:
        return "moderate_10_50"
    if missing_pct <= 90:
        return "high_50_90"
    return "extreme_over_90"


def classify_cardinality(
    unique_count: int,
    unique_ratio: float,
) -> str:
    if unique_count <= 1:
        return "constant"
    if unique_ratio >= 0.98 and unique_count >= 1_000:
        return "identifier_like"
    if unique_count <= 20:
        return "low"
    if unique_count <= 100:
        return "medium"
    if unique_count <= 10_000:
        return "high"
    return "very_high"


def infer_role(
    column: str,
    series: pd.Series,
    unique_count: int,
) -> str:
    if column == JOIN_KEY:
        return "primary_key"
    if column == TARGET_COLUMN:
        return "target"
    if column == TIME_COLUMN:
        return "relative_time"
    if column == AMOUNT_COLUMN:
        return "monetary"

    if column.startswith("card"):
        return "categorical_code"

    if column in {"addr1", "addr2"}:
        return "categorical_code"

    if (
        pd.api.types.is_object_dtype(series.dtype)
        or pd.api.types.is_string_dtype(series.dtype)
        or pd.api.types.is_bool_dtype(series.dtype)
    ):
        return "categorical"

    if unique_count <= 100:
        return "discrete_numeric"

    return "continuous_numeric"

def build_column_profile(
    dataframe: pd.DataFrame,
    table_name: str,
) -> pd.DataFrame:
    row_count = len(dataframe)
    records: list[dict[str, object]] = []

    for column in dataframe.columns:
        series = dataframe[column]

        non_null_count = int(series.notna().sum())
        missing_count = row_count - non_null_count
        missing_pct = (
            100 * missing_count / row_count if row_count else 0.0
        )

        unique_count = int(series.nunique(dropna=True))
        unique_ratio = (
            unique_count / non_null_count if non_null_count else 0.0
        )

        records.append(
            {
                "table": table_name,
                "column": column,
                "dtype": str(series.dtype),
                "row_count": row_count,
                "non_null_count": non_null_count,
                "missing_count": missing_count,
                "missing_pct": missing_pct,
                "unique_count": unique_count,
                "unique_ratio": unique_ratio,
                "missingness_band": classify_missingness(
                    missing_pct
                ),
                "cardinality_band": classify_cardinality(
                    unique_count,
                    unique_ratio,
                ),
                "role_hint": infer_role(
                    column,
                    series,
                    unique_count,
                ),
            }
        )

    return pd.DataFrame.from_records(records)

In [7]:
transaction_profile = build_column_profile(
    transaction,
    "train_transaction",
)
identity_profile = build_column_profile(
    identity,
    "train_identity",
)

column_profile = pd.concat(
    [transaction_profile, identity_profile],
    ignore_index=True,
)

print(f"Profiled columns: {len(column_profile):,}")
print(
    "Transaction columns: "
    f"{len(transaction_profile):,}"
)
print(
    "Identity columns:    "
    f"{len(identity_profile):,}"
)

display(
    column_profile
    .sort_values(
        ["missing_pct", "unique_count"],
        ascending=[False, False],
    )
    .head(25)
)

Profiled columns: 435
Transaction columns: 394
Identity columns:    41


,table,column,dtype,row_count,non_null_count,missing_count,missing_pct,unique_count,unique_ratio,missingness_band,cardinality_band,role_hint
418,train_identity,id_24,float64,144233,4747,139486,96.7088,12,0.0025,extreme_over_90,low,discrete_numeric
419,train_identity,id_25,float64,144233,5132,139101,96.4419,341,0.0664,extreme_over_90,high,continuous_numeric
402,train_identity,id_08,float64,144233,5155,139078,96.4259,94,0.0182,extreme_over_90,medium,discrete_numeric
401,train_identity,id_07,float64,144233,5155,139078,96.4259,84,0.0163,extreme_over_90,medium,discrete_numeric
415,train_identity,id_21,float64,144233,5159,139074,96.4231,490,0.0950,extreme_over_90,high,continuous_numeric
420,train_identity,id_26,float64,144233,5163,139070,96.4204,95,0.0184,extreme_over_90,medium,discrete_numeric
416,train_identity,id_22,float64,144233,5169,139064,96.4162,25,0.0048,extreme_over_90,medium,discrete_numeric
417,train_identity,id_23,str,144233,5169,139064,96.4162,3,0.0006,extreme_over_90,low,categorical
421,train_identity,id_27,str,144233,5169,139064,96.4162,2,0.0004,extreme_over_90,low,categorical
14,train_transaction,dist2,float64,590540,37627,552913,93.6284,1751,0.0465,extreme_over_90,high,continuous_numeric


In [8]:
missingness_order = [
    "complete",
    "low_0_10",
    "moderate_10_50",
    "high_50_90",
    "extreme_over_90",
]

missingness_summary = (
    column_profile
    .assign(
        missingness_band=pd.Categorical(
            column_profile["missingness_band"],
            categories=missingness_order,
            ordered=True,
        )
    )
    .groupby(
        ["table", "missingness_band"],
        observed=False,
    )
    .size()
    .rename("column_count")
    .reset_index()
)

display(missingness_summary)

extremely_sparse = column_profile.loc[
    column_profile["missing_pct"] > 90,
    [
        "table",
        "column",
        "dtype",
        "missing_pct",
        "unique_count",
        "role_hint",
    ],
].sort_values(
    ["table", "missing_pct"],
    ascending=[True, False],
)

print(
    "Columns with more than 90% missingness: "
    f"{len(extremely_sparse)}"
)
display(extremely_sparse.head(30))

,table,missingness_band,column_count
0,train_identity,complete,3
1,train_identity,low_0_10,16
2,train_identity,moderate_10_50,10
3,train_identity,high_50_90,3
4,train_identity,extreme_over_90,9
5,train_transaction,complete,20
6,train_transaction,low_0_10,92
7,train_transaction,moderate_10_50,108
8,train_transaction,high_50_90,172
9,train_transaction,extreme_over_90,2


Columns with more than 90% missingness: 11


,table,column,dtype,missing_pct,unique_count,role_hint
418,train_identity,id_24,float64,96.7088,12,discrete_numeric
419,train_identity,id_25,float64,96.4419,341,continuous_numeric
401,train_identity,id_07,float64,96.4259,84,discrete_numeric
402,train_identity,id_08,float64,96.4259,94,discrete_numeric
415,train_identity,id_21,float64,96.4231,490,continuous_numeric
420,train_identity,id_26,float64,96.4204,95,discrete_numeric
416,train_identity,id_22,float64,96.4162,25,discrete_numeric
417,train_identity,id_23,str,96.4162,3,categorical
421,train_identity,id_27,str,96.4162,2,categorical
14,train_transaction,dist2,float64,93.6284,1751,continuous_numeric


In [9]:
cardinality_order = [
    "constant",
    "low",
    "medium",
    "high",
    "very_high",
    "identifier_like",
]

cardinality_summary = (
    column_profile
    .assign(
        cardinality_band=pd.Categorical(
            column_profile["cardinality_band"],
            categories=cardinality_order,
            ordered=True,
        )
    )
    .groupby(
        ["table", "cardinality_band"],
        observed=False,
    )
    .size()
    .rename("column_count")
    .reset_index()
)

display(cardinality_summary)

constant_columns = column_profile.loc[
    column_profile["cardinality_band"] == "constant",
    [
        "table",
        "column",
        "dtype",
        "missing_pct",
        "unique_count",
    ],
]

identifier_like_columns = column_profile.loc[
    column_profile["cardinality_band"] == "identifier_like",
    [
        "table",
        "column",
        "dtype",
        "missing_pct",
        "unique_count",
        "unique_ratio",
        "role_hint",
    ],
]

print(f"Constant columns:        {len(constant_columns)}")
print(
    "Identifier-like columns: "
    f"{len(identifier_like_columns)}"
)

display(constant_columns)
display(identifier_like_columns)

,table,cardinality_band,column_count
0,train_identity,constant,0
1,train_identity,low,17
2,train_identity,medium,12
3,train_identity,high,10
4,train_identity,very_high,1
5,train_identity,identifier_like,1
6,train_transaction,constant,0
7,train_transaction,low,151
8,train_transaction,medium,99
9,train_transaction,high,122


Constant columns:        0
Identifier-like columns: 2


,table,column,dtype,missing_pct,unique_count


,table,column,dtype,missing_pct,unique_count,unique_ratio,role_hint
0,train_transaction,TransactionID,int64,0.0000,590540,1.0000,primary_key
394,train_identity,TransactionID,int64,0.0000,144233,1.0000,primary_key


In [10]:
selected_risk_fields = [
    "ProductCD",
    "card1",
    "card4",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
]

selected_field_profile = (
    column_profile.loc[
        column_profile["column"].isin(selected_risk_fields),
        [
            "table",
            "column",
            "dtype",
            "missing_pct",
            "unique_count",
            "unique_ratio",
            "cardinality_band",
            "role_hint",
        ],
    ]
    .sort_values(["table", "column"])
    .reset_index(drop=True)
)

display(selected_field_profile)

,table,column,dtype,missing_pct,unique_count,unique_ratio,cardinality_band,role_hint
0,train_identity,DeviceInfo,str,17.7262,1786,0.0151,high,categorical
1,train_identity,DeviceType,str,2.3732,2,0.0000,low,categorical
2,train_transaction,P_emaildomain,str,15.9949,59,0.0001,medium,categorical
3,train_transaction,ProductCD,str,0.0000,5,0.0000,low,categorical
4,train_transaction,R_emaildomain,str,76.7516,60,0.0004,medium,categorical
5,train_transaction,addr1,float64,11.1264,332,0.0006,high,categorical_code
6,train_transaction,addr2,float64,11.1264,74,0.0001,medium,categorical_code
7,train_transaction,card1,int64,0.0000,13553,0.0230,very_high,categorical_code
8,train_transaction,card4,str,0.2670,4,0.0000,low,categorical_code
9,train_transaction,card6,str,0.2660,4,0.0000,low,categorical_code


### Initial modeling implications

The profile supports several early rules:

- columns with extreme missingness require explicit justification
- constant columns provide no model signal
- identifier-like columns should not be used blindly
- high-cardinality categoricals require controlled encoding
- missingness may itself contain fraud-risk information
- identity absence should remain available as an explicit feature

No imputation or feature removal is performed in this section.


## 6. Transaction amount behavior

`TransactionAmt` is the main monetary exposure field.

This section examines:

- data-quality constraints
- distribution and skew
- percentile-based extremes
- legitimate versus fraudulent transaction values
- fraud rate by amount band
- share of total transaction value associated with fraud

Amount bands are descriptive EDA tools, not final production thresholds.


In [11]:
import numpy as np


amount = transaction[AMOUNT_COLUMN]

missing_amount_count = int(amount.isna().sum())
negative_amount_count = int((amount < 0).sum())
zero_amount_count = int((amount == 0).sum())
infinite_amount_count = int(
    np.isinf(amount.dropna().to_numpy()).sum()
)

print(f"Missing amounts:  {missing_amount_count:,}")
print(f"Negative amounts: {negative_amount_count:,}")
print(f"Zero amounts:     {zero_amount_count:,}")
print(f"Infinite amounts: {infinite_amount_count:,}")

assert missing_amount_count == 0
assert negative_amount_count == 0
assert infinite_amount_count == 0

print("Transaction amount quality checks passed.")

Missing amounts:  0
Negative amounts: 0
Zero amounts:     0
Infinite amounts: 0
Transaction amount quality checks passed.


In [12]:
amount_percentiles = [
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    0.999,
    1.00,
]

amount_distribution = (
    amount
    .quantile(amount_percentiles)
    .rename_axis("quantile")
    .rename("transaction_amount")
    .reset_index()
)

amount_distribution["percentile"] = (
    amount_distribution["quantile"] * 100
)

amount_distribution = amount_distribution[
    ["percentile", "transaction_amount"]
]

display(amount_distribution)

print(f"Mean:     {amount.mean():,.4f}")
print(f"Median:   {amount.median():,.4f}")
print(f"Std dev:  {amount.std():,.4f}")
print(f"Skewness: {amount.skew():,.4f}")

,percentile,transaction_amount
0,0.0000,0.2510
1,1.0000,9.2440
2,5.0000,20.0000
3,25.0000,43.3210
4,50.0000,68.7690
5,75.0000,125.0000
6,90.0000,275.2930
7,95.0000,445.0000
8,99.0000,"1,104.0000"
9,99.9000,"2,769.8073"


Mean:     135.0272
Median:   68.7690
Std dev:  239.1625
Skewness: 14.3745


In [13]:
amount_by_target = (
    transaction
    .groupby(TARGET_COLUMN)
    .agg(
        transaction_count=(AMOUNT_COLUMN, "size"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        mean_amount=(AMOUNT_COLUMN, "mean"),
        median_amount=(AMOUNT_COLUMN, "median"),
        maximum_amount=(AMOUNT_COLUMN, "max"),
    )
    .rename(index={0: "legitimate", 1: "fraud"})
)

amount_by_target["transaction_share"] = (
    amount_by_target["transaction_count"]
    / amount_by_target["transaction_count"].sum()
)

amount_by_target["amount_share"] = (
    amount_by_target["total_amount"]
    / amount_by_target["total_amount"].sum()
)

display(amount_by_target)

total_transaction_amount = transaction[AMOUNT_COLUMN].sum()
fraud_transaction_amount = transaction.loc[
    transaction[TARGET_COLUMN] == 1,
    AMOUNT_COLUMN,
].sum()

fraud_amount_share = (
    fraud_transaction_amount / total_transaction_amount
)

print(
    f"Total transaction amount:      "
    f"{total_transaction_amount:,.2f}"
)
print(
    f"Fraudulent transaction amount: "
    f"{fraud_transaction_amount:,.2f}"
)
print(
    f"Fraud share of transaction value: "
    f"{fraud_amount_share:.4%}"
)

,transaction_count,total_amount,mean_amount,median_amount,maximum_amount,transaction_share,amount_share
isFraud,,,,,,,
legitimate,569877,"76,655,103.8750",134.5117,68.5000,"31,937.3910",0.9650,0.9613
fraud,20663,"3,083,844.8600",149.2448,75.0000,"5,191.0000",0.0350,0.0387


Total transaction amount:      79,738,948.73
Fraudulent transaction amount: 3,083,844.86
Fraud share of transaction value: 3.8674%


### Business-readable amount bands

Fixed bands make the results easier to interpret operationally than percentile
groups alone.

These boundaries are exploratory and may later be replaced by payment-product
or currency-specific thresholds.


In [14]:
amount_analysis = transaction[
    [AMOUNT_COLUMN, TARGET_COLUMN]
].copy()

amount_analysis["fraud_amount"] = (
    amount_analysis[AMOUNT_COLUMN]
    * amount_analysis[TARGET_COLUMN]
)

business_band_edges = [
    0,
    25,
    50,
    100,
    200,
    500,
    1_000,
    float("inf"),
]

business_band_labels = [
    "0–<25",
    "25–<50",
    "50–<100",
    "100–<200",
    "200–<500",
    "500–<1,000",
    "1,000+",
]

amount_analysis["business_amount_band"] = pd.cut(
    amount_analysis[AMOUNT_COLUMN],
    bins=business_band_edges,
    labels=business_band_labels,
    right=False,
    include_lowest=True,
)

business_amount_summary = (
    amount_analysis
    .groupby(
        "business_amount_band",
        observed=False,
    )
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
    )
    .reset_index()
)

business_amount_summary["fraud_rate"] = (
    business_amount_summary["fraud_count"]
    / business_amount_summary["transaction_count"]
)

business_amount_summary["transaction_share"] = (
    business_amount_summary["transaction_count"]
    / business_amount_summary["transaction_count"].sum()
)

business_amount_summary["fraud_capture_share"] = (
    business_amount_summary["fraud_count"]
    / business_amount_summary["fraud_count"].sum()
)

business_amount_summary["fraud_value_share"] = (
    business_amount_summary["fraud_amount"]
    / business_amount_summary["fraud_amount"].sum()
)

display(business_amount_summary)

,business_amount_band,transaction_count,fraud_count,total_amount,fraud_amount,fraud_rate,transaction_share,fraud_capture_share,fraud_value_share
0,0–<25,43329,3019,"709,032.2800","46,756.0570",0.0697,0.0734,0.1461,0.0152
1,25–<50,144186,4461,"5,306,136.4680","163,935.9920",0.0309,0.2442,0.2159,0.0532
2,50–<100,160742,4618,"10,608,248.2530","320,696.3210",0.0287,0.2722,0.2235,0.1040
3,100–<200,141813,3999,"17,685,725.7630","526,380.7060",0.0282,0.2401,0.1935,0.1707
4,200–<500,76146,3417,"21,565,359.3380","1,017,352.8130",0.0449,0.1289,0.1654,0.3299
5,"500–<1,000",16744,962,"11,046,874.0810","699,792.6110",0.0575,0.0284,0.0466,0.2269
6,"1,000+",7580,187,"12,817,572.5520","308,930.3600",0.0247,0.0128,0.0090,0.1002


### Amount quantiles

Quantile bands contain approximately equal transaction volumes. They help
distinguish a genuine amount-risk relationship from effects caused by heavily
unequal fixed-band sizes.


In [15]:
amount_analysis["amount_quantile"] = pd.qcut(
    amount_analysis[AMOUNT_COLUMN],
    q=10,
    duplicates="drop",
)

quantile_amount_summary = (
    amount_analysis
    .groupby(
        "amount_quantile",
        observed=False,
    )
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        minimum_amount=(AMOUNT_COLUMN, "min"),
        maximum_amount=(AMOUNT_COLUMN, "max"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
    )
    .reset_index()
)

quantile_amount_summary["fraud_rate"] = (
    quantile_amount_summary["fraud_count"]
    / quantile_amount_summary["transaction_count"]
)

quantile_amount_summary["fraud_capture_share"] = (
    quantile_amount_summary["fraud_count"]
    / quantile_amount_summary["fraud_count"].sum()
)

display(quantile_amount_summary)

,amount_quantile,transaction_count,fraud_count,minimum_amount,maximum_amount,total_amount,fraud_amount,fraud_rate,fraud_capture_share
0,"(0.25, 25.95]",59511,3326,0.2510,25.9500,"1,121,194.6650","54,527.4120",0.0559,0.1610
1,"(25.95, 35.95]",61650,1976,25.9600,35.9500,"1,948,118.5390","61,014.3910",0.0321,0.0956
2,"(35.95, 49.0]",65116,2100,35.9600,49.0000,"2,884,381.4660","91,282.9340",0.0323,0.1016
3,"(49.0, 57.95]",59647,1159,49.0010,57.9500,"3,244,721.4160","61,293.4340",0.0194,0.0561
4,"(57.95, 68.769]",49346,1407,57.9510,68.7580,"3,000,249.1140","86,235.3120",0.0285,0.0681
5,"(68.769, 100.0]",73349,2653,68.7800,100.0000,"6,460,951.8010","229,334.8870",0.0362,0.1284
6,"(100.0, 117.0]",72079,1423,100.0100,117.0000,"8,048,617.0270","160,129.9080",0.0197,0.0689
7,"(117.0, 159.95]",32399,1394,117.0080,159.9500,"4,667,861.8860","200,080.7990",0.0430,0.0675
8,"(159.95, 275.293]",58390,2221,159.9600,275.2930,"12,156,103.2790","463,152.6270",0.0380,0.1075
9,"(275.293, 31937.391]",59053,3004,275.4900,"31,937.3910","36,206,749.5420","1,676,793.1560",0.0509,0.1454


In [16]:
highest_business_band = (
    business_amount_summary
    .sort_values(
        ["fraud_rate", "transaction_count"],
        ascending=[False, False],
    )
    .iloc[0]
)

highest_quantile = (
    quantile_amount_summary
    .sort_values(
        ["fraud_rate", "transaction_count"],
        ascending=[False, False],
    )
    .iloc[0]
)

print(
    "Highest fixed-band fraud rate: "
    f"{highest_business_band['business_amount_band']} "
    f"({highest_business_band['fraud_rate']:.4%}, "
    f"{highest_business_band['transaction_count']:,.0f} transactions)"
)

print(
    "Highest quantile fraud rate: "
    f"{highest_quantile['amount_quantile']} "
    f"({highest_quantile['fraud_rate']:.4%}, "
    f"{highest_quantile['transaction_count']:,.0f} transactions)"
)

Highest fixed-band fraud rate: 0–<25 (6.9676%, 43,329 transactions)
Highest quantile fraud rate: (0.25, 25.95] (5.5889%, 59,511 transactions)


### Initial modeling implications

- the raw transaction amount should be preserved
- a `log1p(TransactionAmt)` feature may help with strong right skew
- amount bands may support monitoring and dashboard segmentation
- transaction count and transaction-value exposure should both be evaluated
- amount alone should not be interpreted as a fraud decision rule
- threshold selection should consider fraud value as well as fraud count


## 7. Chronological behavior and fraud drift

`TransactionDT` represents elapsed time from an undisclosed reference point.
It supports chronological ordering but should not be presented as an exact
calendar timestamp.

This section examines:

- time-field quality
- relative day and week coverage
- transaction-volume drift
- fraud-rate drift
- identity-availability drift
- candidate chronological model splits


In [17]:
time_values = transaction[TIME_COLUMN]

missing_time_count = int(time_values.isna().sum())
negative_time_count = int((time_values < 0).sum())
infinite_time_count = int(
    np.isinf(time_values.dropna().to_numpy()).sum()
)

minimum_transaction_dt = float(time_values.min())
maximum_transaction_dt = float(time_values.max())
elapsed_seconds = (
    maximum_transaction_dt - minimum_transaction_dt
)
elapsed_days = elapsed_seconds / 86_400

source_is_time_sorted = bool(
    time_values.is_monotonic_increasing
)

print(f"Missing time values:  {missing_time_count:,}")
print(f"Negative time values: {negative_time_count:,}")
print(f"Infinite time values: {infinite_time_count:,}")
print(f"Minimum TransactionDT: {minimum_transaction_dt:,.0f}")
print(f"Maximum TransactionDT: {maximum_transaction_dt:,.0f}")
print(f"Observed elapsed days: {elapsed_days:,.2f}")
print(f"Source already sorted: {source_is_time_sorted}")

assert missing_time_count == 0
assert negative_time_count == 0
assert infinite_time_count == 0

print("Transaction time quality checks passed.")

Missing time values:  0
Negative time values: 0
Infinite time values: 0
Minimum TransactionDT: 86,400
Maximum TransactionDT: 15,811,131
Observed elapsed days: 182.00
Source already sorted: True
Transaction time quality checks passed.


In [18]:
time_analysis = transaction[
    [
        JOIN_KEY,
        TIME_COLUMN,
        AMOUNT_COLUMN,
        TARGET_COLUMN,
    ]
].copy()

time_analysis["relative_seconds"] = (
    time_analysis[TIME_COLUMN]
    - minimum_transaction_dt
)

time_analysis["relative_day"] = (
    time_analysis["relative_seconds"] // 86_400
).astype("int32")

time_analysis["relative_week"] = (
    time_analysis["relative_day"] // 7
).astype("int16")

time_analysis["relative_hour_cycle"] = (
    (time_analysis["relative_seconds"] // 3_600) % 24
).astype("int8")

time_analysis["has_identity"] = (
    has_identity.astype("int8")
)

time_analysis["fraud_amount"] = (
    time_analysis[AMOUNT_COLUMN]
    * time_analysis[TARGET_COLUMN]
)

print(
    "Relative day range: "
    f"{time_analysis['relative_day'].min()}–"
    f"{time_analysis['relative_day'].max()}"
)
print(
    "Relative week range: "
    f"{time_analysis['relative_week'].min()}–"
    f"{time_analysis['relative_week'].max()}"
)
print(
    "Relative hour-cycle range: "
    f"{time_analysis['relative_hour_cycle'].min()}–"
    f"{time_analysis['relative_hour_cycle'].max()}"
)

Relative day range: 0–181
Relative week range: 0–25
Relative hour-cycle range: 0–23


### Weekly operational behavior

Weekly summaries reveal whether transaction volume, fraud prevalence, monetary
exposure, or identity-data availability changes through the observation window.

The first and last weeks may be partial periods, so comparisons should consider
transaction support.


In [19]:
weekly_summary = (
    time_analysis
    .groupby("relative_week")
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
        identity_count=("has_identity", "sum"),
        mean_amount=(AMOUNT_COLUMN, "mean"),
    )
    .reset_index()
)

weekly_summary["fraud_rate"] = (
    weekly_summary["fraud_count"]
    / weekly_summary["transaction_count"]
)

weekly_summary["identity_coverage"] = (
    weekly_summary["identity_count"]
    / weekly_summary["transaction_count"]
)

weekly_summary["fraud_value_share"] = (
    weekly_summary["fraud_amount"]
    / weekly_summary["total_amount"]
)

print(f"Observed weekly periods: {len(weekly_summary)}")
display(weekly_summary)

supported_weeks = weekly_summary.loc[
    weekly_summary["transaction_count"] >= 1_000
].copy()

lowest_fraud_week = supported_weeks.loc[
    supported_weeks["fraud_rate"].idxmin()
]

highest_fraud_week = supported_weeks.loc[
    supported_weeks["fraud_rate"].idxmax()
]

print(
    "Lowest supported weekly fraud rate: "
    f"week {lowest_fraud_week['relative_week']:.0f}, "
    f"{lowest_fraud_week['fraud_rate']:.4%}, "
    f"{lowest_fraud_week['transaction_count']:,.0f} transactions"
)

print(
    "Highest supported weekly fraud rate: "
    f"week {highest_fraud_week['relative_week']:.0f}, "
    f"{highest_fraud_week['fraud_rate']:.4%}, "
    f"{highest_fraud_week['transaction_count']:,.0f} transactions"
)

Observed weekly periods: 26


,relative_week,transaction_count,fraud_count,total_amount,fraud_amount,identity_count,mean_amount,fraud_rate,identity_coverage,fraud_value_share
0,0,27596,804,"3,614,446.7170","115,303.6990",7493,130.9772,0.0291,0.2715,0.0319
1,1,28463,717,"3,622,050.2620","102,518.3570",11471,127.2547,0.0252,0.4030,0.0283
2,2,35701,869,"4,725,933.1300","103,375.8590",17934,132.3754,0.0243,0.5023,0.0219
3,3,34906,724,"4,164,729.5170","101,438.1320",19028,119.3127,0.0207,0.5451,0.0244
4,4,24539,889,"3,246,536.6860","121,564.4370",5725,132.3011,0.0362,0.2333,0.0374
5,5,20919,803,"2,882,970.2670","122,015.6610",3833,137.8159,0.0384,0.1832,0.0423
6,6,20564,823,"2,768,204.4540","104,885.8080",3252,134.6141,0.0400,0.1581,0.0379
7,7,19761,862,"2,651,959.3400","118,201.7440",3511,134.2017,0.0436,0.1777,0.0446
8,8,21432,927,"3,038,368.2370","127,460.3340",3820,141.7678,0.0433,0.1782,0.0420
9,9,22534,996,"3,049,335.3040","134,119.4330",3961,135.3215,0.0442,0.1758,0.0440


Lowest supported weekly fraud rate: week 3, 2.0741%, 34,906 transactions
Highest supported weekly fraud rate: week 16, 5.0716%, 21,078 transactions


### Equal-volume chronological periods

Ten chronological periods provide a stable comparison because each period
contains approximately the same number of transactions.

This helps distinguish genuine drift from fluctuations caused primarily by
unequal sample sizes.


In [20]:
time_analysis["chronological_period"] = pd.qcut(
    time_analysis[TIME_COLUMN],
    q=10,
    labels=[
        "P01",
        "P02",
        "P03",
        "P04",
        "P05",
        "P06",
        "P07",
        "P08",
        "P09",
        "P10",
    ],
    duplicates="drop",
)

chronological_period_summary = (
    time_analysis
    .groupby(
        "chronological_period",
        observed=False,
    )
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        minimum_transaction_dt=(TIME_COLUMN, "min"),
        maximum_transaction_dt=(TIME_COLUMN, "max"),
        minimum_relative_day=("relative_day", "min"),
        maximum_relative_day=("relative_day", "max"),
        mean_amount=(AMOUNT_COLUMN, "mean"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
        identity_count=("has_identity", "sum"),
    )
    .reset_index()
)

chronological_period_summary["fraud_rate"] = (
    chronological_period_summary["fraud_count"]
    / chronological_period_summary["transaction_count"]
)

chronological_period_summary["identity_coverage"] = (
    chronological_period_summary["identity_count"]
    / chronological_period_summary["transaction_count"]
)

chronological_period_summary["fraud_capture_share"] = (
    chronological_period_summary["fraud_count"]
    / chronological_period_summary["fraud_count"].sum()
)

chronological_period_summary["fraud_value_share"] = (
    chronological_period_summary["fraud_amount"]
    / chronological_period_summary["fraud_amount"].sum()
)

display(chronological_period_summary)

first_period = chronological_period_summary.iloc[0]
last_period = chronological_period_summary.iloc[-1]

print(
    "Fraud-rate change, P01 to P10: "
    f"{first_period['fraud_rate']:.4%} -> "
    f"{last_period['fraud_rate']:.4%}"
)

print(
    "Identity-coverage change, P01 to P10: "
    f"{first_period['identity_coverage']:.4%} -> "
    f"{last_period['identity_coverage']:.4%}"
)

,chronological_period,transaction_count,fraud_count,minimum_transaction_dt,maximum_transaction_dt,minimum_relative_day,maximum_relative_day,mean_amount,total_amount,fraud_amount,identity_count,fraud_rate,identity_coverage,fraud_capture_share,fraud_value_share
0,P01,59054,1631,86400,1360999,0,14,129.2562,"7,633,098.0260","229,663.4780",20364,0.0276,0.3448,0.0789,0.0745
1,P02,59054,1195,1361005,2310138,14,25,124.1928,"7,334,078.9600","150,090.9470",32475,0.0202,0.5499,0.0578,0.0487
2,P03,59054,2200,2310165,3864159,25,43,135.7951,"8,019,244.3490","309,764.7760",13485,0.0373,0.2284,0.1065,0.1004
3,P04,59054,2532,3864166,5592303,43,63,136.2820,"8,047,994.8950","345,308.1240",10180,0.0429,0.1724,0.1225,0.1120
4,P05,59054,2336,5592304,7306520,63,83,133.5870,"7,888,847.4950","305,381.3840",11644,0.0396,0.1972,0.1131,0.0990
5,P06,59054,2094,7306535,8745772,83,100,146.8357,"8,671,234.2850","337,581.9680",9168,0.0355,0.1552,0.1013,0.1095
6,P07,59054,2550,8745798,10437996,100,119,136.2855,"8,048,202.8450","439,538.0740",12595,0.0432,0.2133,0.1234,0.1425
7,P08,59054,2061,10438003,12192842,119,140,132.9769,"7,852,815.6980","356,581.7730",10553,0.0349,0.1787,0.0997,0.1156
8,P09,59054,1851,12192900,13990904,140,160,138.4039,"8,173,304.7460","268,480.9840",10360,0.0313,0.1754,0.0896,0.0871
9,P10,59054,2213,13990941,15811131,160,181,136.6567,"8,070,127.4360","341,453.3520",13409,0.0375,0.2271,0.1071,0.1107


Fraud-rate change, P01 to P10: 2.7619% -> 3.7474%
Identity-coverage change, P01 to P10: 34.4837% -> 22.7063%


### Candidate chronological model split

The internal labeled dataset is divided by chronological row order:

- earliest 70%: training
- next 15%: validation
- latest 15%: test

This is preferable to a random split because a production fraud model scores
future transactions rather than randomly sampled historical transactions.


In [21]:
chronological_index = (
    transaction[TIME_COLUMN]
    .sort_values(kind="stable")
    .index
)

row_count = len(transaction)
train_end_position = int(row_count * 0.70)
validation_end_position = int(row_count * 0.85)

split_assignment = pd.Series(
    index=transaction.index,
    dtype="object",
)

split_assignment.loc[
    chronological_index[:train_end_position]
] = "train"

split_assignment.loc[
    chronological_index[
        train_end_position:validation_end_position
    ]
] = "validation"

split_assignment.loc[
    chronological_index[validation_end_position:]
] = "test"

time_analysis["candidate_split"] = pd.Categorical(
    split_assignment,
    categories=["train", "validation", "test"],
    ordered=True,
)

split_summary = (
    time_analysis
    .groupby(
        "candidate_split",
        observed=False,
    )
    .agg(
        transaction_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        minimum_transaction_dt=(TIME_COLUMN, "min"),
        maximum_transaction_dt=(TIME_COLUMN, "max"),
        minimum_relative_day=("relative_day", "min"),
        maximum_relative_day=("relative_day", "max"),
        total_amount=(AMOUNT_COLUMN, "sum"),
        fraud_amount=("fraud_amount", "sum"),
        identity_count=("has_identity", "sum"),
    )
    .reset_index()
)

split_summary["transaction_share"] = (
    split_summary["transaction_count"]
    / split_summary["transaction_count"].sum()
)

split_summary["fraud_rate"] = (
    split_summary["fraud_count"]
    / split_summary["transaction_count"]
)

split_summary["identity_coverage"] = (
    split_summary["identity_count"]
    / split_summary["transaction_count"]
)

display(split_summary)

train_max_time = split_summary.loc[
    split_summary["candidate_split"] == "train",
    "maximum_transaction_dt",
].item()

validation_min_time = split_summary.loc[
    split_summary["candidate_split"] == "validation",
    "minimum_transaction_dt",
].item()

validation_max_time = split_summary.loc[
    split_summary["candidate_split"] == "validation",
    "maximum_transaction_dt",
].item()

test_min_time = split_summary.loc[
    split_summary["candidate_split"] == "test",
    "minimum_transaction_dt",
].item()

assert split_assignment.notna().all()
assert split_summary["transaction_count"].sum() == row_count
assert train_max_time <= validation_min_time
assert validation_max_time <= test_min_time

print("Chronological split contract checks passed.")

,candidate_split,transaction_count,fraud_count,minimum_transaction_dt,maximum_transaction_dt,minimum_relative_day,maximum_relative_day,total_amount,fraud_amount,identity_count,transaction_share,fraud_rate,identity_coverage
0,train,413378,14538,86400,10437996,0,119,"55,642,700.8550","2,117,328.7510",109911,0.7000,0.0352,0.2659
1,validation,88581,3042,10438003,13151840,119,151,"11,947,493.6940","496,907.5880",15654,0.1500,0.0343,0.1767
2,test,88581,3083,13151880,15811131,151,181,"12,148,754.1860","469,608.5210",18668,0.1500,0.0348,0.2107


Chronological split contract checks passed.


### Initial modeling implications

- model validation must preserve chronological ordering
- fraud prevalence should be compared across train, validation, and test periods
- identity-data availability may drift over time
- production monitoring should track both feature drift and fraud-rate drift
- threshold performance may change as transaction composition changes
- random cross-validation would overstate deployment realism

`relative_hour_cycle` is anchored to the dataset's unknown reference point. It
must not be described as a verified local clock hour.


## 8. Fraud patterns across selected risk dimensions

This section compares transaction volume, fraud frequency, and monetary
exposure across selected product-risk dimensions.

The analysis uses a minimum-support threshold so that very small categories are
not interpreted as stable risk segments.

Selected dimensions:

- product type
- card network and card type
- payer and recipient email domains
- identity-data availability
- device type

These are descriptive associations and must not be interpreted as causal fraud
rules.


In [22]:
MINIMUM_CATEGORY_SUPPORT = 1_000


def summarize_risk_dimension(
    dataframe: pd.DataFrame,
    dimension: str,
    *,
    minimum_support: int = MINIMUM_CATEGORY_SUPPORT,
) -> pd.DataFrame:
    required_columns = {
        dimension,
        TARGET_COLUMN,
        AMOUNT_COLUMN,
    }

    missing_columns = required_columns - set(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"{dimension} analysis is missing columns: "
            + ", ".join(sorted(missing_columns))
        )

    working = dataframe[
        [dimension, TARGET_COLUMN, AMOUNT_COLUMN]
    ].copy()

    working["category"] = (
        working[dimension]
        .astype("string")
        .fillna("<MISSING>")
    )

    working["fraud_amount"] = (
        working[AMOUNT_COLUMN]
        * working[TARGET_COLUMN]
    )

    summary = (
        working
        .groupby("category", dropna=False)
        .agg(
            transaction_count=(TARGET_COLUMN, "size"),
            fraud_count=(TARGET_COLUMN, "sum"),
            total_amount=(AMOUNT_COLUMN, "sum"),
            fraud_amount=("fraud_amount", "sum"),
            mean_amount=(AMOUNT_COLUMN, "mean"),
        )
        .reset_index()
    )

    summary["dimension"] = dimension

    summary["fraud_rate"] = (
        summary["fraud_count"]
        / summary["transaction_count"]
    )

    summary["transaction_share"] = (
        summary["transaction_count"]
        / summary["transaction_count"].sum()
    )

    summary["fraud_capture_share"] = (
        summary["fraud_count"]
        / summary["fraud_count"].sum()
    )

    total_fraud_amount = summary["fraud_amount"].sum()

    summary["fraud_value_share"] = (
        summary["fraud_amount"] / total_fraud_amount
        if total_fraud_amount
        else 0.0
    )

    summary["support_status"] = np.where(
        summary["transaction_count"] >= minimum_support,
        "supported",
        "low_support",
    )

    return summary[
        [
            "dimension",
            "category",
            "transaction_count",
            "fraud_count",
            "fraud_rate",
            "transaction_share",
            "fraud_capture_share",
            "total_amount",
            "fraud_amount",
            "fraud_value_share",
            "mean_amount",
            "support_status",
        ]
    ].sort_values(
        ["support_status", "fraud_rate", "transaction_count"],
        ascending=[False, False, False],
    )

In [23]:
risk_base = transaction[
    [
        JOIN_KEY,
        TARGET_COLUMN,
        AMOUNT_COLUMN,
        "ProductCD",
        "card4",
        "card6",
        "P_emaildomain",
        "R_emaildomain",
    ]
].copy()

risk_base["identity_availability"] = np.where(
    has_identity,
    "has_identity",
    "no_identity",
)

device_lookup = identity[
    [
        JOIN_KEY,
        "DeviceType",
    ]
].copy()

risk_base = risk_base.merge(
    device_lookup,
    on=JOIN_KEY,
    how="left",
    validate="one_to_one",
)

assert len(risk_base) == len(transaction)
assert risk_base[JOIN_KEY].is_unique

risk_dimensions = [
    "ProductCD",
    "card4",
    "card6",
    "P_emaildomain",
    "R_emaildomain",
    "identity_availability",
    "DeviceType",
]

risk_summaries = {
    dimension: summarize_risk_dimension(
        risk_base,
        dimension,
    )
    for dimension in risk_dimensions
}

print(
    f"Minimum supported category size: "
    f"{MINIMUM_CATEGORY_SUPPORT:,}"
)
print(f"Risk dimensions analyzed: {len(risk_summaries)}")

Minimum supported category size: 1,000
Risk dimensions analyzed: 7


In [24]:
low_cardinality_dimensions = [
    "ProductCD",
    "card4",
    "card6",
    "identity_availability",
    "DeviceType",
]

low_cardinality_summary = pd.concat(
    [
        risk_summaries[dimension]
        for dimension in low_cardinality_dimensions
    ],
    ignore_index=True,
)

display(
    low_cardinality_summary[
        [
            "dimension",
            "category",
            "transaction_count",
            "fraud_count",
            "fraud_rate",
            "transaction_share",
            "fraud_capture_share",
            "fraud_value_share",
            "support_status",
        ]
    ]
)

,dimension,category,transaction_count,fraud_count,fraud_rate,transaction_share,fraud_capture_share,fraud_value_share,support_status
0,ProductCD,C,68519,8008,0.1169,0.1160,0.3876,0.1269,supported
1,ProductCD,S,11628,686,0.0590,0.0197,0.0332,0.0141,supported
2,ProductCD,H,33024,1574,0.0477,0.0559,0.0762,0.0800,supported
3,ProductCD,R,37699,1426,0.0378,0.0638,0.0690,0.1129,supported
4,ProductCD,W,439670,8969,0.0204,0.7445,0.4341,0.6662,supported
5,card4,discover,6651,514,0.0773,0.0113,0.0249,0.0590,supported
6,card4,visa,384767,13373,0.0348,0.6516,0.6472,0.6478,supported
7,card4,mastercard,189217,6496,0.0343,0.3204,0.3144,0.2775,supported
8,card4,american express,8328,239,0.0287,0.0141,0.0116,0.0139,supported
9,card4,<MISSING>,1577,41,0.0260,0.0027,0.0020,0.0018,supported


### Email-domain segments

Email domains have medium cardinality. Only categories meeting the minimum
support threshold are ranked as interpretable segments.

Rare domains may later be grouped into an `OTHER` category for baseline
modeling.


In [25]:
for email_dimension in [
    "P_emaildomain",
    "R_emaildomain",
]:
    email_summary = risk_summaries[email_dimension]

    supported_email_summary = (
        email_summary.loc[
            email_summary["support_status"] == "supported"
        ]
        .sort_values(
            ["fraud_rate", "transaction_count"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )

    low_support_count = int(
        (
            email_summary["support_status"]
            == "low_support"
        ).sum()
    )

    print(f"\n{email_dimension}")
    print(
        "Supported categories: "
        f"{len(supported_email_summary)}"
    )
    print(
        "Low-support categories: "
        f"{low_support_count}"
    )

    display(
        supported_email_summary[
            [
                "category",
                "transaction_count",
                "fraud_count",
                "fraud_rate",
                "transaction_share",
                "fraud_capture_share",
                "fraud_value_share",
            ]
        ].head(15)
    )


P_emaildomain
Supported categories: 20
Low-support categories: 40


,category,transaction_count,fraud_count,fraud_rate,transaction_share,fraud_capture_share,fraud_value_share
0,outlook.com,5096,482,0.0946,0.0086,0.0233,0.0132
1,hotmail.com,45250,2396,0.0530,0.0766,0.1160,0.0534
2,gmail.com,228355,9943,0.0435,0.3867,0.4812,0.4526
3,icloud.com,6267,197,0.0314,0.0106,0.0095,0.0072
4,comcast.net,7888,246,0.0312,0.0134,0.0119,0.0407
5,<MISSING>,94456,2790,0.0295,0.1599,0.1350,0.1388
6,bellsouth.net,1909,53,0.0278,0.0032,0.0026,0.0041
7,live.com,3041,84,0.0276,0.0051,0.0041,0.0030
8,anonymous.com,36998,859,0.0232,0.0627,0.0416,0.0396
9,yahoo.com,100934,2297,0.0228,0.1709,0.1112,0.1550



R_emaildomain
Supported categories: 10
Low-support categories: 51


,category,transaction_count,fraud_count,fraud_rate,transaction_share,fraud_capture_share,fraud_value_share
0,outlook.com,2507,414,0.1651,0.0042,0.0200,0.0089
1,icloud.com,1398,180,0.1288,0.0024,0.0087,0.0075
2,gmail.com,57147,6811,0.1192,0.0968,0.3296,0.2081
3,hotmail.com,27509,2140,0.0778,0.0466,0.1036,0.0348
4,yahoo.com,11842,610,0.0515,0.0201,0.0295,0.0283
5,aol.com,3701,129,0.0349,0.0063,0.0062,0.0063
6,anonymous.com,20529,598,0.0291,0.0348,0.0289,0.0144
7,<MISSING>,453249,9436,0.0208,0.7675,0.4567,0.6810
8,comcast.net,1812,21,0.0116,0.0031,0.0010,0.0007
9,yahoo.com.mx,1508,16,0.0106,0.0026,0.0008,0.0004


In [26]:
dimension_overview_records = []

for dimension, summary in risk_summaries.items():
    supported = summary.loc[
        summary["support_status"] == "supported"
    ].copy()

    supported = supported.sort_values(
        ["fraud_rate", "transaction_count"],
        ascending=[False, False],
    )

    missing_rows = summary.loc[
        summary["category"] == "<MISSING>"
    ]

    if supported.empty:
        top_category = None
        top_transaction_count = 0
        top_fraud_rate = np.nan
    else:
        top_row = supported.iloc[0]
        top_category = top_row["category"]
        top_transaction_count = int(
            top_row["transaction_count"]
        )
        top_fraud_rate = float(top_row["fraud_rate"])

    if missing_rows.empty:
        missing_transaction_share = 0.0
        missing_fraud_rate = np.nan
    else:
        missing_row = missing_rows.iloc[0]
        missing_transaction_share = float(
            missing_row["transaction_share"]
        )
        missing_fraud_rate = float(
            missing_row["fraud_rate"]
        )

    dimension_overview_records.append(
        {
            "dimension": dimension,
            "category_count": len(summary),
            "supported_category_count": len(supported),
            "highest_supported_category": top_category,
            "highest_supported_count": top_transaction_count,
            "highest_supported_fraud_rate": top_fraud_rate,
            "missing_transaction_share": (
                missing_transaction_share
            ),
            "missing_fraud_rate": missing_fraud_rate,
        }
    )

risk_dimension_overview = pd.DataFrame(
    dimension_overview_records
)

display(risk_dimension_overview)

,dimension,category_count,supported_category_count,highest_supported_category,highest_supported_count,highest_supported_fraud_rate,missing_transaction_share,missing_fraud_rate
0,ProductCD,5,5,C,68519,0.1169,0.0000,NaN
1,card4,5,5,discover,6651,0.0773,0.0027,0.0260
2,card6,5,3,credit,148986,0.0668,0.0027,0.0248
3,P_emaildomain,60,20,outlook.com,5096,0.0946,0.1599,0.0295
4,R_emaildomain,61,10,outlook.com,2507,0.1651,0.7675,0.0208
5,identity_availability,2,2,has_identity,144233,0.0785,0.0000,NaN
6,DeviceType,3,3,mobile,55645,0.1017,0.7616,0.0210


### Initial modeling implications

- low-cardinality product and card fields are suitable baseline categoricals
- email domains require missing-value handling and rare-category grouping
- category-level fraud rates must use minimum-support safeguards
- identity availability should be retained as an explicit feature
- `DeviceType` can be used as a small categorical feature
- missing `DeviceType` includes both absent identity records and identity rows
  where the device type itself is unavailable
- group fraud rates are monitoring signals, not standalone block rules
- any target-based encoding must be learned inside the training split only
